# Template masks against Loghub's annotated templates

## Config

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from logstore.templating import to_template

# both files are from the HPC folder of https://github.com/logpai/loghub
SAMPLE_PATH = ROOT / "data" / "loghub" / "HPC_2k.log_structured.csv"
TEMPLATES_PATH = ROOT / "data" / "loghub" / "HPC_2k.log_templates.csv"

pd.set_option("display.max_colwidth", 120)

## Load Loghub's annotated 2,000-line sample

In [2]:
sample = pd.read_csv(SAMPLE_PATH, keep_default_na=False)
annotated = pd.read_csv(TEMPLATES_PATH, keep_default_na=False)
print(f"{len(sample):,} lines, {sample.EventId.nunique()} annotated templates in the sample, {len(annotated)} in the template list")
sample.head()

2,000 lines, 46 annotated templates in the sample, 46 in the template list


,LineId,LogId,Node,Component,State,Time,Flag,Content,EventId,EventTemplate
0,1,134681,node-246,unix.hw,state_change.unavailable,1077804742,1,Component State Change: Component \042SCSI-WWID:01000010:6005-08b4-0001-00c6-0006-3000-003d-0000\042 is in the unava...,E13,Component State Change: Component <*> is in the unavailable state (HWID=<*>)
1,2,350766,node-109,unix.hw,state_change.unavailable,1084680778,1,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3180),E13,Component State Change: Component <*> is in the unavailable state (HWID=<*>)
2,3,344518,node-246,unix.hw,state_change.unavailable,1084270955,1,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=5089),E13,Component State Change: Component <*> is in the unavailable state (HWID=<*>)
3,4,344448,node-153,unix.hw,state_change.unavailable,1084270952,1,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=4088),E13,Component State Change: Component <*> is in the unavailable state (HWID=<*>)
4,5,366633,node-200,unix.hw,state_change.unavailable,1085100843,1,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538),E13,Component State Change: Component <*> is in the unavailable state (HWID=<*>)


## Apply the repo's masks

In [3]:
sample["template"] = sample.Content.map(to_template)
sample["key"] = sample.Component + " | " + sample.State + " | " + sample.template
sample[["Content", "template", "EventTemplate"]].drop_duplicates("template").head(10)

,Content,template,EventTemplate
0,Component State Change: Component \042SCSI-WWID:01000010:6005-08b4-0001-00c6-0006-3000-003d-0000\042 is in the unava...,Component State Change: Component <Q> is in the unavailable state (HWID=<N>),Component State Change: Component <*> is in the unavailable state (HWID=<*>)
12,clusterAddMember (command 1902),clusterAddMember (command <N>),clusterAddMember (command <*>)
14,risBoot (command 1903),risBoot (command <N>),risBoot (command <*>)
15,bootGenvmunix (command 1903),bootGenvmunix (command <N>),bootGenvmunix (command <*>)
16,boot (command 1911),boot (command <N>),boot (command <*>)
20,wait (command 1975),wait (command <N>),wait (command <*>)
22,halt (command 1982),halt (command <N>),halt (command <*>)
153,Command has completed successfully,Command has completed successfully,Command has completed successfully
163,Targeting domains:node-D3 and nodes:node-[104-127] child of command 1963,Targeting domains:<NODE> and nodes:<NODE> child of command <N>,Targeting domains:node-D<*> and nodes:node-[<*>-<*>] child of command <*>
165,Targeting domains:node-D7 and nodes:node-[224\ 228\ 232],Targeting domains:<NODE> and nodes:<NODE>,Targeting domains:node-D<*>and nodes:node-<*>


## Grouping accuracy
Share of lines whose group holds exactly the lines of their annotated template.

In [4]:
def grouping_accuracy(predicted, annotated):
    frame = pd.DataFrame({"predicted": predicted, "annotated": annotated})
    predicted_size = frame.groupby("predicted").predicted.transform("size")
    annotated_size = frame.groupby("annotated").annotated.transform("size")
    shared_size = frame.groupby(["predicted", "annotated"]).predicted.transform("size")
    # the two groups hold the same lines when their overlap is as large as each of them
    return ((shared_size == predicted_size) & (shared_size == annotated_size)).mean()


pd.DataFrame(
    {
        "groups": [sample.template.nunique(), sample.key.nunique()],
        "grouping_accuracy": [
            grouping_accuracy(sample.template, sample.EventId),
            grouping_accuracy(sample.key, sample.EventId),
        ],
    },
    index=["masked text", "component, state and masked text"],
)

,groups,grouping_accuracy
masked text,56,0.9310
"component, state and masked text",62,0.8005


## Groups that merge more than one annotated template

In [5]:
pairs = sample.groupby(["EventId", "EventTemplate", "template"]).size().rename("lines").reset_index()
merged = pairs[pairs.groupby("template").EventId.transform("nunique") > 1]
print(f"{merged.template.nunique()} groups merge more than one annotated template")
merged

1 groups merge more than one annotated template


,EventId,EventTemplate,template,lines
8,E17,Fan speeds ( <*> <*> <*> **** <*> <*> ),Fan speeds ( <N> <N> <N> <N> <N> <N> ),79
9,E18,Fan speeds ( <*> <*> <*> <*> <*> <*> ),Fan speeds ( <N> <N> <N> <N> <N> <N> ),11


## Annotated templates that the masks split into several groups

In [6]:
split = pairs[pairs.groupby("EventId").template.transform("nunique") > 1]
print(f"{split.EventId.nunique()} annotated templates are split, covering {split.lines.sum()} lines")
split.sort_values(["EventId", "lines"], ascending=[True, False])

2 annotated templates are split, covering 48 lines


,EventId,EventTemplate,template,lines
34,E40,ServerFileSystem: ServerFileSystem domain <*> is full,ServerFileSystem: ServerFileSystem domain root<N>_local is full,11
35,E40,ServerFileSystem: ServerFileSystem domain <*> is full,ServerFileSystem: ServerFileSystem domain root<N>_tmp is full,1
49,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain root<N>_domain is no longer served by node <NODE>,14
51,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain root<N>_local is no longer served by node <NODE>,5
52,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain root<N>_tmp is no longer served by node <NODE>,4
56,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>,4
47,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain cluster_usr_backup is no longer served by node <NODE>,2
54,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain root_domain is no longer served by node <NODE>,2
46,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain cluster_root_backup is no longer served by node <NODE>,1
48,E9,ClusterFileSystem: ServerFileSystem domain <*> is no longer served by node node-<*>,ClusterFileSystem: ServerFileSystem domain cluster_var_backup is no longer served by node <NODE>,1


## Annotated templates that appear under more than one component or state

In [7]:
by_key = sample.groupby(["EventId", "EventTemplate", "Component", "State"]).size().rename("lines").reset_index()
by_key[by_key.groupby("EventId").State.transform("size") > 1]

,EventId,EventTemplate,Component,State,lines
13,E21,Link error,switch_module,bcast-error,9
14,E21,Link error,switch_module,error,4
17,E24,Link in reset,switch_module,bcast-error,2
18,E24,Link in reset,switch_module,error,7
19,E25,Link ok,switch_module,bcast-error,1
20,E25,Link ok,switch_module,error,12
28,E32,not responding,domain,status,3
29,E32,not responding,node,status,57
35,E38,running,node,status,133
36,E38,running,partition,status,29
